# Relatório Lab 5 — Câmera Estéreo, Calibração e Vídeo 3D

**Disciplina:** ESZA019 — Visão Computacional  
**Professor:** Celso Kurashima  
**Grupo:** Grupo 2  
**Integrantes:** Cesar de Jesus; Mariana Chiara; Vinicius de Marchi 
**Data de realização do experimento:** 08/07/2026  
**Data de publicação do relatório:** 08/07/2026  

---

## Resumo

Neste laboratório foi montado e calibrado um sistema de câmera estéreo usando duas webcams conectadas a um computador Linux. O experimento envolveu a captura de pares de imagens de um tabuleiro de calibração, a detecção dos cantos internos do padrão, a calibração individual das câmeras, a calibração estéreo do conjunto, a retificação das imagens e a geração de uma visualização 3D em anáglifo. Ao final, também foi gravado um vídeo em formato `.mp4` contendo o resultado anáglifo obtido a partir das duas webcams.

A calibração utilizou **15 pares válidos** de imagens com resolução **640 x 480 pixels**. O erro RMS obtido foi de **0.2244** para a câmera esquerda, **0.2140** para a câmera direita e **0.6385** para a calibração estéreo. Esses valores indicam uma calibração consistente para um experimento prático com webcams de laboratório.

## 1. Introdução

A visão estereoscópica utiliza duas imagens capturadas a partir de pontos de vista ligeiramente diferentes para inferir informações de profundidade. O princípio é semelhante ao funcionamento da visão binocular humana: objetos mais próximos apresentam maior deslocamento relativo entre as imagens esquerda e direita, enquanto objetos mais distantes apresentam menor deslocamento.

Para que essa comparação seja geometricamente válida, é necessário calibrar o sistema de câmeras. A calibração estima os parâmetros internos de cada câmera, os coeficientes de distorção das lentes e a relação geométrica entre as duas câmeras. Depois disso, as imagens podem ser retificadas para que pontos correspondentes fiquem alinhados aproximadamente na mesma linha horizontal, facilitando o cálculo de disparidade.

Neste laboratório, o objetivo foi construir uma câmera estéreo com duas webcams, capturar imagens de calibração com um tabuleiro, estimar os parâmetros do par estéreo e utilizar esses parâmetros para visualizar uma cena ao vivo e gravada em formato 3D anáglifo.

## 2. Objetivos

Os objetivos do experimento foram:

1. montar um par estéreo usando duas webcams fixas;
2. capturar entre 10 e 15 pares de imagens de um tabuleiro de calibração;
3. detectar automaticamente os cantos internos do tabuleiro nas imagens esquerda e direita;
4. calibrar individualmente cada câmera;
5. estimar a rotação e a translação entre as câmeras;
6. salvar os parâmetros de calibração em arquivo XML;
7. gerar visualização estéreo ao vivo em anáglifo;
8. gravar um vídeo 3D em formato `.mp4`;
9. analisar os parâmetros obtidos e discutir uma aplicação real da câmera estéreo no Trabalho Final.

## 3. Fundamentação teórica

### 3.1 Modelo de câmera

Uma câmera pode ser aproximada pelo modelo pinhole, no qual um ponto 3D do mundo é projetado em um plano de imagem. A transformação entre coordenadas do mundo e coordenadas da imagem depende de parâmetros extrínsecos e intrínsecos.

Os **parâmetros intrínsecos** descrevem propriedades internas de cada câmera, como distância focal em pixels e centro óptico. Eles são representados pela matriz:

$$
K =
egin{bmatrix}
f_x & 0 & c_x \
0 & f_y & c_y \
0 & 0 & 1
\end{bmatrix}
$$

em que $f_x$ e $f_y$ são as distâncias focais em pixels, e $(c_x, c_y)$ é o ponto principal da imagem.

Além da matriz intrínseca, a calibração estima os coeficientes de distorção da lente, normalmente associados à distorção radial e tangencial. Esses coeficientes corrigem deformações causadas pela lente, especialmente em regiões próximas às bordas da imagem.

### 3.2 Calibração estéreo

No sistema estéreo, além dos parâmetros de cada câmera, é necessário estimar a posição relativa entre elas. Essa relação é dada por:

- **R**: matriz de rotação entre as câmeras;
- **T**: vetor de translação entre as câmeras;
- **E**: matriz essencial, associada à geometria epipolar em coordenadas normalizadas;
- **F**: matriz fundamental, associada à geometria epipolar em coordenadas de pixel.

A calibração estéreo permite obter também as matrizes de retificação **R1**, **R2**, **P1**, **P2** e a matriz **Q**, usada para reprojeção 3D a partir da disparidade.

### 3.3 Disparidade e profundidade

Após a retificação, pontos correspondentes nas duas imagens ficam, idealmente, na mesma linha horizontal. A diferença entre as coordenadas horizontais desses pontos é chamada de disparidade:

$$
d = x_L - x_R
$$

A profundidade é inversamente proporcional à disparidade. Em uma forma simplificada:

$$
Z pprox rac{fB}{d}
$$

em que $Z$ é a profundidade, $f$ é a distância focal, $B$ é a baseline entre as câmeras e $d$ é a disparidade. Assim, quanto maior a disparidade, mais próximo o objeto está do sistema estéreo.

### 3.4 Anáglifo

O anáglifo é uma forma de visualização 3D que combina canais de cor das imagens esquerda e direita em uma única imagem. Normalmente, um canal vermelho é associado a uma câmera e os canais verde/azul à outra. Com óculos vermelho-ciano, cada olho recebe predominantemente uma das imagens, produzindo a percepção de profundidade.

## 4. Materiais e ambiente utilizado

O experimento foi realizado em ambiente Linux, com Python e OpenCV. A estrutura utilizada foi:

- computador com Linux/Ubuntu;
- ambiente Conda `CV26`;
- duas webcams conectadas ao mesmo computador;
- tabuleiro de calibração;
- biblioteca OpenCV para captura, detecção de cantos, calibração e retificação;
- biblioteca NumPy para manipulação de matrizes;
- scripts Python adaptados para o experimento do Grupo 2.

As duas webcams foram mantidas fixas uma em relação à outra durante a captura, calibração, visualização ao vivo e gravação. Esse cuidado é essencial, porque a calibração estéreo só é válida enquanto a posição relativa das câmeras permanecer a mesma.

## 5. Procedimentos experimentais

### 5.1 Captura das imagens de calibração

Primeiro, foram capturados pares de imagens do tabuleiro com as duas webcams. O script utilizado foi:

```bash
python3 codigos/capture_images_abc.py
```

O programa abriu as duas câmeras, verificou os identificadores das webcams e salvou pares de imagens quando o tabuleiro foi detectado simultaneamente nas duas visões.

Foram obtidos **15 pares válidos** de imagens. As imagens foram salvas em:

```text
dados/data/stereoL/
dados/data/stereoR/
```

A figura abaixo mostra um exemplo de par estéreo capturado.

![Par estéreo de calibração](figuras/par_estereo_exemplo.png)

### 5.2 Detecção dos cantos do tabuleiro

Após a captura, os cantos internos do tabuleiro foram detectados usando a função `cv2.findChessboardCorners`. A detecção dos cantos é necessária porque a calibração compara as posições conhecidas dos pontos no mundo com suas posições observadas na imagem.

O padrão usado no experimento foi de **6 x 8 cantos internos**, conforme registrado no arquivo de resultados. A imagem abaixo mostra um exemplo de detecção dos cantos nas duas câmeras.

![Cantos detectados](figuras/cantos_detectados_exemplo.png)

### 5.3 Calibração das câmeras e calibração estéreo

A calibração foi executada com:

```bash
python3 codigos/calibrate_abc.py
```

O procedimento realizado pelo script pode ser resumido em quatro etapas:

1. leitura dos pares de imagens esquerda e direita;
2. detecção dos cantos internos do tabuleiro;
3. calibração individual de cada câmera com `cv2.calibrateCamera`;
4. calibração estéreo com `cv2.stereoCalibrate` e retificação com `cv2.stereoRectify`.

Ao final, foram gerados os arquivos:

```text
resultados/calibracao_resultados_abc.txt
resultados/stereo_params_abc.xml
resultados/parametros_calibracao.npz
```

O arquivo XML contém os parâmetros necessários para reutilizar a calibração nas etapas seguintes do laboratório.

### 5.4 Visualização estéreo ao vivo

Depois da calibração, foi executado o programa de visualização estéreo ao vivo:

```bash
python3 codigos/movie3d_abc.py
```

Esse script lê o arquivo XML de calibração, abre novamente as duas webcams, aplica a retificação das imagens e gera uma visualização em anáglifo. A retificação é importante porque transforma as imagens para uma configuração em que as linhas epipolares ficam alinhadas horizontalmente, tornando mais simples a comparação entre a imagem esquerda e a direita.

### 5.5 Gravação do vídeo 3D

Por fim, o vídeo em anáglifo foi gravado com:

```bash
python3 codigos/movie3d_abc_gravacao.py
```

O arquivo gerado foi:

```text
resultados/video3d_anaglifo_abc.mp4
```

A figura abaixo apresenta dois frames extraídos do resultado em anáglifo.

![Frames do anáglifo](figuras/anaglifo_resultados.png)

## 6. Resultados obtidos

A tabela a seguir resume os principais resultados da calibração.

| Parâmetro | Valor obtido |
|---|---:|
| Tamanho da imagem | 640 x 480 pixels |
| Padrão do tabuleiro | 6 x 8 cantos internos |
| Tamanho do quadrado usado | 1.0 |
| Pares válidos usados | 15 |
| Erro RMS da câmera esquerda | 0.224408 |
| Erro RMS da câmera direita | 0.214041 |
| Erro RMS da calibração estéreo | 0.638500 |
| Baseline $\|T\|$ | 1.885733 |

O valor de baseline está na mesma unidade adotada para o tamanho do quadrado do tabuleiro. Como o código utilizou `square_size = 1.0`, a baseline está expressa em unidades de quadrado do tabuleiro, e não diretamente em centímetros. Para obter a baseline em centímetros, seria necessário substituir `square_size` pelo tamanho real medido de cada quadrado.

## 7. Parâmetros de calibração obtidos

A seguir são apresentados os principais parâmetros estimados.

### M1 — matriz intrinseca da camera esquerda

```text
[[672.39759524   0.         339.4300318 ]
 [  0.         670.94209783 244.19256014]
 [  0.           0.           1.        ]]
```

### D1 — coeficientes de distorcao da esquerda

```text
[[-0.00219936 -0.13712282 -0.00246572  0.0097713   0.36297575]]
```

### M2 — matriz intrinseca da camera direita

```text
[[671.03378912   0.         325.05301367]
 [  0.         671.36513437 263.41725511]
 [  0.           0.           1.        ]]
```

### D2 — coeficientes de distorcao da direita

```text
[[ 0.00338912 -0.25456771 -0.00576592  0.00941688  0.56625559]]
```

### R — rotacao entre as cameras

```text
[[ 0.99798984  0.01404888  0.06179736]
 [-0.01361699  0.99987987 -0.00740441]
 [-0.06189396  0.00654804  0.99806125]]
```

### T — translacao entre as cameras

```text
[[ 1.88396256]
 [-0.05184198]
 [-0.06312751]]
```

### E — matriz essencial

```text
[[ 0.0023491   0.06278046 -0.05220889]
 [ 0.05360528 -0.01322312 -1.88421114]
 [ 0.02608387  1.88446456 -0.01074594]]
```

### F — matriz fundamental

```text
[[ 6.92519153e-08  1.85479613e-06 -1.51134025e-03]
 [ 1.57951493e-06 -3.90473322e-07 -3.77719779e-02]
 [ 7.74147565e-05  3.68597059e-02  1.00000000e+00]]
```

### R1 — retificacao esquerda

```text
[[ 0.99949939 -0.0136719   0.02853143]
 [ 0.01376256  0.99990084 -0.00298339]
 [-0.02848781  0.00337456  0.99958844]]
```

### R2 — retificacao direita

```text
[[ 0.99906133 -0.02749169 -0.03347638]
 [ 0.02759801  0.99961541  0.00271796]
 [ 0.03338879 -0.00363929  0.99943581]]
```

### P1 — nova matriz de projecao esquerda

```text
[[719.74798613   0.         338.92874908   0.        ]
 [  0.         719.74798613 252.48764801   0.        ]
 [  0.           0.           1.           0.        ]]
```

### P2 — nova matriz de projecao direita

```text
[[7.19747986e+02 0.00000000e+00 3.38928749e+02 1.35725227e+03]
 [0.00000000e+00 7.19747986e+02 2.52487648e+02 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00 0.00000000e+00]]
```

### Q — matriz de reprojecao 3D

```text
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00 -3.38928749e+02]
 [ 0.00000000e+00  1.00000000e+00  0.00000000e+00 -2.52487648e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  7.19747986e+02]
 [ 0.00000000e+00  0.00000000e+00 -5.30297869e-01  0.00000000e+00]]
```

## 8. Parâmetros salvos no arquivo XML

O arquivo `stereo_params_abc.xml` armazena os parâmetros necessários para reaproveitar a calibração sem repetir todo o processo. No experimento, foram salvos:

```text
image_width, image_height, board_cols, board_rows, square_size, num_valid_pairs,
rms_left, rms_right, rms_stereo, M1, D1, M2, D2, R, T, E, F,
R1, R2, P1, P2, Q, roi1, roi2
```

O significado prático desses parâmetros é:

| Parâmetro | Significado |
|---|---|
| `image_width`, `image_height` | resolução das imagens utilizadas na calibração |
| `board_cols`, `board_rows` | número de cantos internos do tabuleiro |
| `square_size` | escala física adotada para os pontos do tabuleiro |
| `num_valid_pairs` | número de pares usados efetivamente na calibração |
| `rms_left`, `rms_right`, `rms_stereo` | erros médios da calibração |
| `M1`, `M2` | matrizes intrínsecas das câmeras esquerda e direita |
| `D1`, `D2` | coeficientes de distorção das lentes |
| `R`, `T` | rotação e translação entre as câmeras |
| `E`, `F` | matrizes essencial e fundamental da geometria epipolar |
| `R1`, `R2` | rotações de retificação |
| `P1`, `P2` | novas matrizes de projeção após retificação |
| `Q` | matriz de reprojeção 3D a partir da disparidade |
| `roi1`, `roi2` | regiões válidas após retificação |

## 9. Análise dos resultados

A calibração individual das câmeras apresentou erros RMS próximos de **0.2244** para a câmera esquerda e **0.2140** para a direita. Esses valores são baixos para uma calibração prática com webcams, indicando que os cantos do tabuleiro foram detectados de maneira coerente e que o modelo de câmera conseguiu explicar bem as observações.

O erro RMS da calibração estéreo foi **0.6385**, maior que os erros individuais. Isso é esperado, pois a calibração estéreo envolve uma restrição mais forte: além de cada câmera estar calibrada individualmente, é necessário que a relação geométrica entre as duas câmeras explique simultaneamente os pares de imagens.

As matrizes intrínsecas obtidas para as duas câmeras apresentam valores de distância focal semelhantes, próximos de 670 pixels antes da retificação. Isso sugere que as webcams tinham campos de visão parecidos. Os centros ópticos estimados também ficaram próximos ao centro da imagem de 640 x 480 pixels, embora com pequenos deslocamentos, o que é normal em câmeras reais.

O vetor de translação estimado foi:

```text
T = [1.88396256, -0.05184198, -0.06312751]^T
```

A componente dominante está no eixo horizontal, o que é coerente com a montagem de duas webcams lado a lado. As componentes vertical e em profundidade são pequenas, indicando que as câmeras estavam aproximadamente alinhadas, mas não perfeitamente paralelas. A matriz de rotação também ficou próxima da identidade, com pequenas rotações associadas aos desalinhamentos físicos da montagem.

A baseline obtida foi **1.8857** unidades de quadrado. Como o tamanho real do quadrado não foi informado ao algoritmo, essa unidade é relativa. Mesmo assim, o valor é suficiente para a geração de retificação, disparidade e visualização anáglifa.

A visualização em anáglifo apresentou percepção de separação entre os pontos de vista esquerdo e direito. A qualidade da percepção 3D depende da iluminação, da textura dos objetos, da estabilidade das câmeras, da qualidade da calibração e do uso de óculos anáglifos adequados.

## 10. Comparação entre resultado ao vivo e resultado gravado

Durante a visualização ao vivo, a percepção do anáglifo tende a ser mais imediata, pois a resposta do sistema acompanha diretamente a movimentação da cena. No vídeo gravado, o efeito 3D permanece, mas pode haver perdas associadas à compressão do arquivo `.mp4`, variações de taxa de quadros e pequenas diferenças de sincronização entre as câmeras.

De forma geral, o resultado gravado foi suficiente para documentar o funcionamento da câmera estéreo. Porém, a visualização ao vivo tende a facilitar a percepção de profundidade, especialmente quando há movimento controlado de objetos próximos e distantes.

### Percepção individual dos integrantes

Como os nomes ainda serão preenchidos, deixamos abaixo os campos para completar antes da entrega final:

- **Integrante 1:** inserir percepção individual sobre o resultado ao vivo e o vídeo gravado.
- **Integrante 2:** inserir percepção individual sobre o resultado ao vivo e o vídeo gravado.
- **Integrante 3:** inserir percepção individual sobre o resultado ao vivo e o vídeo gravado.

Sugestão de redação para cada integrante:

> Durante o teste ao vivo, foi possível perceber a separação entre as imagens esquerda e direita no anáglifo, principalmente quando objetos eram posicionados em diferentes profundidades. No vídeo gravado, o efeito permaneceu visível, mas com menor nitidez devido à compressão e à visualização em tela. A estabilidade da montagem das webcams foi essencial para manter a coerência do resultado.

## 11. Aplicação da câmera estéreo no Trabalho Final

A câmera estéreo construída neste experimento pode ser aplicada em um Trabalho Final envolvendo percepção de profundidade, medição de distância, navegação visual, rastreamento de objetos ou interação homem-máquina. A principal vantagem em relação a uma única câmera é que o sistema estéreo permite estimar a distância relativa dos objetos usando disparidade.

Uma aplicação real possível é um sistema de detecção e análise de objetos em 3D. O fluxo geral seria:

```text
Duas webcams fixas
        ↓
Captura sincronizada aproximada
        ↓
Correção de distorção e retificação estéreo
        ↓
Cálculo de disparidade
        ↓
Estimativa de profundidade
        ↓
Detecção/rastreamento de objeto
        ↓
Tomada de decisão baseada em posição e distância
```

Em uma aplicação de robótica móvel, por exemplo, a câmera estéreo poderia ajudar a estimar a distância até obstáculos. Em uma aplicação de rastreamento de objetos, ela poderia fornecer não apenas a posição do objeto na imagem, mas também uma estimativa de profundidade. Em uma aplicação de postura, gestos ou interação com o usuário, o sistema poderia diferenciar movimentos próximos e distantes da câmera, fornecendo informação espacial adicional.

Do ponto de vista técnico, a calibração obtida neste laboratório fornece os elementos centrais para essa aplicação: as matrizes intrínsecas de cada câmera, a relação extrínseca entre as webcams, as matrizes de retificação e a matriz de reprojeção 3D. Dessa forma, o mesmo sistema pode ser reaproveitado para calcular disparidade e profundidade em cenas reais, desde que as câmeras não sejam movidas após a calibração.

## 12. Conclusões

O Lab 5 permitiu montar e calibrar um sistema de visão estéreo utilizando duas webcams. A captura dos pares de imagens do tabuleiro foi bem-sucedida, com 15 pares válidos utilizados na calibração. Os erros RMS obtidos foram compatíveis com um experimento de laboratório, indicando boa detecção dos cantos e uma calibração consistente.

A calibração individual estimou os parâmetros intrínsecos e as distorções de cada câmera. A calibração estéreo estimou a rotação e a translação entre as webcams, permitindo a retificação das imagens e a geração de visualização em anáglifo. A gravação do vídeo 3D confirmou que os parâmetros salvos puderam ser reutilizados para processar imagens ao vivo.

A principal limitação do experimento foi a dependência da estabilidade física das câmeras. Qualquer alteração na posição relativa entre as webcams exige nova calibração. Além disso, a qualidade da percepção 3D depende de boa iluminação, textura na cena, sincronização adequada e escolha correta dos parâmetros de disparidade.

Mesmo com essas limitações, o experimento demonstrou de forma prática como a calibração estéreo fornece a base para percepção de profundidade e aplicações de visão 3D.

## 13. Referências

- Richard Szeliski. *Computer Vision: Algorithms and Applications*. 2nd ed., Springer, 2022.
- Gonzalez, R. C.; Woods, R. E. *Digital Image Processing*. 4th ed., Pearson, 2018.
- Documentação do OpenCV — Camera Calibration and 3D Reconstruction.
- Material da disciplina ESZA019 — Visão Computacional, UFABC, 2026.2.
- Roteiro do Lab 5 — Câmera Estéreo, Calibração e Vídeo 3D.

## 14. Reprodutibilidade

Para reproduzir o experimento em Linux, com o ambiente `CV26` ativo, executar:

```bash
conda activate CV26
python3 codigos/capture_images_abc.py
python3 codigos/calibrate_abc.py
python3 codigos/movie3d_abc.py
python3 codigos/movie3d_abc_gravacao.py
```

Os arquivos esperados após a execução são:

```text
resultados/stereo_params_abc.xml
resultados/calibracao_resultados_abc.txt
resultados/video3d_anaglifo_abc.mp4
figuras/anaglifo_frame_01.png
figuras/anaglifo_frame_02.png
```

Os dados brutos de calibração estão em:

```text
dados/data/stereoL/
dados/data/stereoR/
dados/data/corners_detected/
```

In [ ]:
# Célula opcional para conferir os principais resultados da calibração.
# Execute esta célula no Jupyter caso queira validar os arquivos do relatório.

from pathlib import Path

arquivo = Path("resultados/calibracao_resultados_abc.txt")
print(arquivo.read_text(encoding="utf-8")[:2000])

In [ ]:
# Célula opcional para visualizar alguns resultados no Jupyter.

from IPython.display import Image, display

for img in [
    "figuras/par_estereo_exemplo.png",
    "figuras/cantos_detectados_exemplo.png",
    "figuras/anaglifo_resultados.png",
]:
    print(img)
    display(Image(filename=img))